# Symbol Exemplar Library Builder

One-time setup. Reads the reference PDFs (`Types_of_signals.pdf`,
`Types_of_signals_ncr.pdf`) and saves each exemplar image as
`library/<class_name>/exemplar_NN.png`, plus a `library/index.json` that
maps every exemplar to its class name and source.

Run this once. The output `library/` folder gets reused across every SIP
you process.

**Scope for this iteration**: extract everything we can find. The matching
notebook will pick which classes to actually use.


## Setup

```
pip install pymupdf pillow numpy
```

In [ ]:
from pathlib import Path
import json
import re
import io

import fitz  # PyMuPDF
from PIL import Image

# Resolve project root. Notebook lives at notebooks/02_build_library.ipynb
# but cwd may be either the repo root or the notebooks dir depending on how
# the kernel was launched.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

REFS_DIR = PROJECT_ROOT / "data" / "refs"
LIB_OUT = PROJECT_ROOT / "library"


def _find_pdf(name: str) -> Path:
    """Look for a reference PDF in data/refs/ first, then PROJECT_ROOT (legacy
    location). Returns the first match."""
    for candidate in (REFS_DIR / name, PROJECT_ROOT / name):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{name} not found in {REFS_DIR} or {PROJECT_ROOT}. "
        f"Drop the reference PDF in {REFS_DIR}/ to proceed."
    )


TYPES_PDF     = _find_pdf("Types_of_signals.pdf")
TYPES_NCR_PDF = _find_pdf("Types_of_signals_ncr.pdf")
LIB_OUT.mkdir(exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Types PDF    : {TYPES_PDF}")
print(f"NCR PDF      : {TYPES_NCR_PDF}")
print(f"Library out  : {LIB_OUT}")


## 1. Extract from `Types_of_signals.pdf`

This reference PDF is a 3-column table: `S.No | TYPE OF SIGNALS | Ref:Image`.
Each row's image is an embedded JPEG with known placement coordinates.

The extraction:
1. For each page, collect text spans and image placements with bboxes.
2. Find serial-number markers in the leftmost column (x ≈ 116).
3. For each row, the type-name spans are in the middle column (x in 150..470)
   within the row's Y range.
4. The exemplar images are in the right column at the same Y range.
5. Group images by row → one or more exemplars per class.

Some rows have zero images (entries that say "to be represented same as ...").
Those don't get folders; we map them to their parent class in a later cell.

In [ ]:
def slugify(s):
    """Make a class name into a filesystem-safe identifier."""
    s = s.strip()
    # Collapse internal whitespace, replace non-alphanumeric with underscores
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_]", "", s)
    return s


def extract_rows_from_types_pdf(pdf_path):
    """Yield {sno, type_name, image_xrefs, image_bboxes} for each row."""
    doc = fitz.open(pdf_path)
    rows = []
    for page_idx, page in enumerate(doc):
        spans = []
        for b in page.get_text("dict")["blocks"]:
            if b.get("type") != 0: continue
            for line in b["lines"]:
                for span in line["spans"]:
                    t = span["text"].strip()
                    if t:
                        spans.append({"text": t, "bbox": span["bbox"]})

        images = []
        for img in page.get_images(full=True):
            xref = img[0]
            for r in page.get_image_rects(xref):
                images.append({"xref": xref, "bbox": (r.x0, r.y0, r.x1, r.y1)})

        # Serial numbers are pure-integer text in the leftmost column (x≈116).
        # Header has page numbers at x≈418 — exclude those.
        sn_spans = [s for s in spans
                    if re.fullmatch(r"\d{1,2}", s["text"])
                    and 110 < s["bbox"][0] < 130
                    and s["bbox"][1] > 50]
        sn_spans.sort(key=lambda s: s["bbox"][1])

        for i, sn in enumerate(sn_spans):
            y_start = sn["bbox"][1] - 2
            y_end = sn_spans[i+1]["bbox"][1] - 2 if i+1 < len(sn_spans) else float("inf")

            type_spans = [s for s in spans
                          if y_start <= s["bbox"][1] < y_end
                          and 150 < s["bbox"][0] < 470
                          and s is not sn]
            type_spans.sort(key=lambda s: (s["bbox"][1], s["bbox"][0]))
            type_name = " ".join(s["text"] for s in type_spans).strip()
            if not type_name: continue

            row_images = [img for img in images
                          if y_start <= img["bbox"][1] < y_end]

            rows.append({
                "page": page_idx + 1,
                "sno": int(sn["text"]),
                "type_name": type_name,
                "image_xrefs": [img["xref"] for img in row_images],
                "image_bboxes": [img["bbox"] for img in row_images],
            })
    doc.close()
    return rows


rows = extract_rows_from_types_pdf(TYPES_PDF)
print(f"Found {len(rows)} rows in {TYPES_PDF.name}")
n_with_images = sum(1 for r in rows if r["image_xrefs"])
print(f"  {n_with_images} have at least one exemplar image")
print(f"  {len(rows) - n_with_images} are placeholder rows (use 'represent same as...' notes)")

print("\nFirst 10 rows:")
for r in rows[:10]:
    print(f"  {r['sno']:2d}: {r['type_name'][:40]:40s} -> {len(r['image_xrefs'])} images")

In [ ]:
# Save each exemplar to library/<class_name>/exemplar_NN.png
saved = []  # list of {class_name, exemplar_path, source}

doc = fitz.open(TYPES_PDF)
for r in rows:
    if not r["image_xrefs"]:
        continue
    class_name = slugify(r["type_name"])
    class_dir = LIB_OUT / class_name
    class_dir.mkdir(exist_ok=True)

    for i, xref in enumerate(r["image_xrefs"]):
        info = doc.extract_image(xref)
        ext = info["ext"]  # 'jpeg', 'png', etc.
        img_bytes = info["image"]

        # Save as PNG so downstream is consistent regardless of source format.
        img = Image.open(io.BytesIO(img_bytes))
        out_path = class_dir / f"exemplar_{i+1:02d}.png"
        img.save(out_path, format="PNG")

        saved.append({
            "class_name": class_name,
            "type_name": r["type_name"],
            "exemplar_path": str(out_path.relative_to(LIB_OUT)),
            "source_pdf": TYPES_PDF.name,
            "source_page": r["page"],
            "source_xref": xref,
            "size": [img.width, img.height],
        })
doc.close()

print(f"Saved {len(saved)} exemplars from {TYPES_PDF.name}")
print(f"Distinct classes: {len({s['class_name'] for s in saved})}")

## 2. Extract from `Types_of_signals_ncr.pdf`

This reference PDF has a different format: free-flowing pages where each
labeled row contains one *or more* example images from real SIPs (which is
extremely useful — these match what we'll see in production better than
the clean reference drawings).

The structure: a left-column text label followed by a right-column stack of
images. Same Y-coordinate matching approach works, but the column layout
is slightly different.

In [ ]:
def extract_rows_from_ncr_pdf(pdf_path):
    """Yield {label, image_xrefs, image_bboxes} for the NCR reference PDF.

    Different from Types_of_signals.pdf:
    - No serial number column; labels themselves mark each row.
    - Labels are in the left column (roughly x < 400).
    - Images are in the right column (roughly x > 400).
    - Some labels span multiple lines.
    """
    doc = fitz.open(pdf_path)
    rows = []
    for page_idx, page in enumerate(doc):
        spans = []
        for b in page.get_text("dict")["blocks"]:
            if b.get("type") != 0: continue
            for line in b["lines"]:
                for span in line["spans"]:
                    t = span["text"].strip()
                    if t:
                        spans.append({"text": t, "bbox": span["bbox"]})

        images = []
        for img in page.get_images(full=True):
            xref = img[0]
            for r in page.get_image_rects(xref):
                images.append({"xref": xref, "bbox": (r.x0, r.y0, r.x1, r.y1)})

        # Heuristic: any span in the left column (x < 400) starting a "row"
        # in the table. We treat each unique Y-bucket of left-column text as
        # a row marker, then group images that fall within that row's Y range.
        left_spans = [s for s in spans if s["bbox"][0] < 400]
        # Drop title-like spans at the very top
        left_spans = [s for s in left_spans if s["bbox"][1] > 80]
        if not left_spans:
            continue

        # Cluster by Y to handle multi-line labels
        left_spans.sort(key=lambda s: s["bbox"][1])
        row_groups = []
        for s in left_spans:
            if row_groups and s["bbox"][1] - row_groups[-1][-1]["bbox"][3] < 20:
                row_groups[-1].append(s)
            else:
                row_groups.append([s])

        for i, group in enumerate(row_groups):
            label = " ".join(s["text"] for s in group)
            y_start = group[0]["bbox"][1] - 5
            y_end = (row_groups[i+1][0]["bbox"][1] - 5
                     if i+1 < len(row_groups) else float("inf"))

            row_images = [img for img in images
                          if y_start <= img["bbox"][1] < y_end]
            if not row_images:
                continue

            rows.append({
                "page": page_idx + 1,
                "label": label,
                "image_xrefs": [img["xref"] for img in row_images],
                "image_bboxes": [img["bbox"] for img in row_images],
            })
    doc.close()
    return rows


ncr_rows = extract_rows_from_ncr_pdf(TYPES_NCR_PDF)
print(f"Found {len(ncr_rows)} rows in {TYPES_NCR_PDF.name}")
for r in ncr_rows[:15]:
    print(f"  p{r['page']} {r['label'][:55]:55s} -> {len(r['image_xrefs'])} images")

In [ ]:
# Save NCR exemplars. Each row's label may map to multiple existing classes
# from Types_of_signals.pdf, but for simplicity we save each row as its own
# class (slugified). A later step can de-duplicate or alias these.
doc = fitz.open(TYPES_NCR_PDF)
for r in ncr_rows:
    class_name = slugify(r["label"])
    if not class_name:
        continue
    class_dir = LIB_OUT / class_name
    class_dir.mkdir(exist_ok=True)

    # Find the highest existing exemplar number so we don't overwrite
    existing = sorted(class_dir.glob("exemplar_*.png"))
    start_idx = len(existing) + 1

    for i, xref in enumerate(r["image_xrefs"]):
        info = doc.extract_image(xref)
        img_bytes = info["image"]
        img = Image.open(io.BytesIO(img_bytes))

        out_path = class_dir / f"exemplar_{start_idx + i:02d}.png"
        img.save(out_path, format="PNG")

        saved.append({
            "class_name": class_name,
            "type_name": r["label"],
            "exemplar_path": str(out_path.relative_to(LIB_OUT)),
            "source_pdf": TYPES_NCR_PDF.name,
            "source_page": r["page"],
            "source_xref": xref,
            "size": [img.width, img.height],
        })
doc.close()

print(f"Total saved exemplars now: {len(saved)}")
print(f"Total distinct classes: {len({s['class_name'] for s in saved})}")

## 3. Save the index

Write `library/index.json` summarizing every exemplar. Downstream code
(matching cells in the main pipeline notebook) reads this to know what
classes exist, where their exemplar images are, and where they came from.

In [ ]:
# Group by class for easier inspection
from collections import defaultdict
by_class = defaultdict(list)
for s in saved:
    by_class[s["class_name"]].append(s)

index = {
    "classes": sorted(by_class.keys()),
    "exemplars_by_class": {
        cls: [{"path": e["exemplar_path"],
               "source_pdf": e["source_pdf"],
               "source_page": e["source_page"],
               "size": e["size"]}
              for e in entries]
        for cls, entries in by_class.items()
    },
    "type_names": {
        # Map the slug back to the original human-readable label
        cls: entries[0]["type_name"]
        for cls, entries in by_class.items()
    },
}

with open(LIB_OUT / "index.json", "w") as f:
    json.dump(index, f, indent=2)

print(f"Wrote {LIB_OUT / 'index.json'}")
print(f"  {len(index['classes'])} classes")
print(f"  {sum(len(v) for v in index['exemplars_by_class'].values())} exemplars total")
print()
print("Class breakdown:")
for cls in sorted(by_class.keys()):
    print(f"  {cls:50s}: {len(by_class[cls])} exemplar(s)")

## 4. Spot check

Display a few exemplars to confirm the extraction worked. Random sample
across classes.

In [ ]:
from IPython.display import Image as IPyImage, display

# Show one exemplar from each of up to 8 classes
sample_classes = sorted(by_class.keys())[:8]
for cls in sample_classes:
    print(f"\n{cls}: {index['type_names'][cls]}")
    paths = [e["path"] for e in index["exemplars_by_class"][cls][:1]]
    for p in paths:
        display(IPyImage(str(LIB_OUT / p)))

## What's next

You now have `library/` with one folder per signal class, each containing
PNG exemplars. Move on to `notebooks/03_detect_symbols.ipynb`, which
consumes this library to detect signal symbols on a SIP:

- **Template matching** for distinctive simple shapes (KM marker, S/B box,
  BSLB, GL gate-lodge box).
- **DINOv2 nearest-neighbor** for the chained-ellipse signal posts
  (Distant, Home, Starter, Shunt, Calling-on).

The library is built once and reused across every SIP. If you add new SIPs
or want to refine class names later, re-run this notebook.
